# Sigma1 Stability Analysis

Mirrors the sigma0 stability notebook but for **sigma1** (ISIs 1, 2, 4).

| Section | Question |
|---------|----------|
| A | Per-ISI stability vs n_experiments |
| B | Combined-ISI stability |
| C | Which ISI subsets are most informative? |

In [1]:
import sys, os, yaml, torch, random
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from collections import defaultdict
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from pathlib import Path
from scipy.spatial.distance import pdist
from tqdm.notebook import trange, tqdm

sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path
from utls.plotting import ensure_dir
from utls.loading import (load_results_with_exclusion_2, move_sequences_to_used,
                           load_results_with_exclusion_no_dropping)
from utls.runners_v2 import run_experiment_scores, make_noise_schedule
from utls.runners_utils import *
from encoders import *
from utls.toy_experiments import (
    make_isi_n_block_experiment, make_toy_experiment_list, make_multi_isi_toy_experiments,
)
from utls.sigma_fitting import log_mid, make_grid, auc_to_dprime


In [2]:
def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path

def median_pairwise_distance(X, metric="euclidean", n_samples=500, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(X.shape[0], size=min(n_samples, X.shape[0]), replace=False)
    return float(np.median(pdist(X[idx], metric=metric)))

CONFIG_PATH = (
    "/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/"
    "model_yamls/v14_three-stage-testing/run_000004.yaml"
)
model_cfg, model_cfg_path = load_config(CONFIG_PATH)
print(model_cfg)


{'results_root': '/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory', 'tag': 'slurm', 'experiment': {'is_multi': True, 'n_seqs': 36, 'n_samples': 50, 'which_task': 0}, 'metric': 'cosine', 'noise_model': {'name': 'three-regime', 'sigma0_min': 1.0, 'sigma0_max': 40.0, 'sigma1_min': 0.1, 'sigma1_max': 40.0, 'sigma2_min': 0.0005, 'sigma2_max': 20.0, 't_step': 5}, 'fitting': {'n_grid': 5, 'n_mc': 32, 'n_refine_iters': 4, 'n_experiments_per_isi': 5, 'k_stimuli_per_exp': 5}, 'run_id': 'run_000004', 'representation': {'type': 'resnet50', 'layer': 'layer4', 'time_avg': False}}


In [3]:
exp_cfg    = model_cfg["experiment"]
which_task = exp_cfg["which_task"]
is_multi   = exp_cfg["is_multi"]
which_isi  = exp_cfg.get("which_isi", None)
isis       = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]
metric     = model_cfg["metric"]
noise_cfg  = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]
t_step     = noise_cfg["t_step"]
repr_cfg   = model_cfg["representation"]
time_avg   = repr_cfg["time_avg"]
encoder_type = repr_cfg["type"]
layer      = repr_cfg.get("layer", None)
pc_dims    = repr_cfg.get("pc_dims", None)

exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)
human_curve  = compute_human_curve(human_runs, is_multi, which_isi)
time_avg_tag = "time_avg" if time_avg else "nontime_avg"
print("ISIs:", isis)
print("Human d':", human_curve)


/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/runners_utils.py:209: RuntimeWarning: Mean of empty slice
  dprimes.append(np.nanmean(aucs))


ISIs: [0, 1, 2, 4, 8, 16, 32, 64]
Human d': [3.40068125 2.94542279 2.40502508 2.13577075 1.97601741 1.91533975
 1.78086152 1.59277116]


In [4]:
NN_ENCODERS  = {"kell2018", "resnet50"}
encoder_task = "word_speaker_audioset" if encoder_type in NN_ENCODERS else "audioset"
encoder_cfg  = dict(
    encoder_type=encoder_type, model_name=encoder_type, task=encoder_task,
    statistics_dict=statistics_set.statistics, model_params=model_params,
    pc_dims=pc_dims, sr=20000, duration=2.0, rms_level=0.05,
    time_avg=time_avg, device="cuda",
)
if encoder_type in NN_ENCODERS: encoder_cfg["layer"] = layer
if encoder_type == "texture":   encoder_cfg["pc_dims"] = pc_dims

encoder_name = make_encoder_name(encoder_cfg)
encoder      = build_encoder(encoder_cfg)
X            = encode_stimuli(encoder, all_files)
X_np         = X.detach().cpu().numpy()
print("Shape:", X_np.shape, " NaN?", torch.isnan(X).any().item())

d50 = median_pairwise_distance(X_np, metric="cosine")
print(f"d50 = {d50:.6f}")

param_bounds = {
    "sigma0": (noise_cfg["sigma0_min"],         noise_cfg["sigma0_max"]),
    "sigma1": (noise_cfg["sigma1_min"] * d50,   noise_cfg["sigma1_max"] * d50),
    "sigma2": (noise_cfg["sigma2_min"] * d50,   noise_cfg["sigma2_max"] * d50),
}
for k, v in param_bounds.items():
    print(f"  {k}: ({v[0]:.6f}, {v[1]:.6f})")

stimulus_pool = sorted({s for seq in exp_list for s in seq})
print(f"Stimulus pool: {len(stimulus_pool)}")


LOADING FROM /orcd/data/jhm/001/om2/bjmedina/models/cochdnn/model_directories/resnet50_word_speaker_audioset/
=> loading checkpoint '/orcd/data/jhm/001/om2/bjmedina/models/cochdnn/model_checkpoints/audio_rep_training_cochleagram_1/standard_training_word_and_audioset_and_speaker_decay_lr/542752d7-9849-49ff-b84a-6758a81585b4/5_checkpoint.pt'
=> loaded checkpoint '/orcd/data/jhm/001/om2/bjmedina/models/cochdnn/model_checkpoints/audio_rep_training_cochleagram_1/standard_training_word_and_audioset_and_speaker_decay_lr/542752d7-9849-49ff-b84a-6758a81585b4/5_checkpoint.pt' (epoch 6)
Shape: (80, 186368)  NaN? False
d50 = 0.479312
  sigma0: (1.000000, 40.000000)
  sigma1: (0.047931, 19.172478)
  sigma2: (0.000240, 9.586239)
Stimulus pool: 80


In [5]:
# ---- fix other sigmas at geometric means ----
sigma0_fixed = 13.0 #log_mid(*param_bounds['sigma0'])
sigma2_fixed = log_mid(*param_bounds['sigma2'])
print(f'Fixed sigma0 = {sigma0_fixed:.6f}')
print(f'Fixed sigma2 = {sigma2_fixed:.6f}')

isi_to_hc_idx = {isi_val: i for i, isi_val in enumerate(isis)}
N_MC      = 128
N_PER_DIM = 10
K_STIM    = 5   # enough for ISI 1-4 (need >= ISI+1)

sigma1_grid = make_grid(param_bounds['sigma1'][0], param_bounds['sigma1'][1],
                        N_PER_DIM, spacing='log')
print('sigma1 grid:', [f'{v:.5f}' for v in sigma1_grid])

print('\nHuman d\' targets:')
for isi_val in [1, 2, 4]:
    idx = isi_to_hc_idx[isi_val]
    print(f'  ISI {isi_val}: {human_curve[idx]:.4f}')


Fixed sigma0 = 15.000000
Fixed sigma2 = 0.033892
sigma1 grid: ['0.04793', '0.08364', '0.14595', '0.25467', '0.44440', '0.77546', '1.35315', '2.36120', '4.12023', '7.18968']

Human d' targets:
  ISI 1: 2.9840
  ISI 2: 2.3835
  ISI 4: 2.1909


In [6]:
def run_sigma_sweep(sigma_name, sigma_grid, fixed_sigmas, exps_by_isi,
                    isi_to_hc_idx, human_curve, N_MC, t_step, noise_mode,
                    metric, X, name_to_idx, base_seed=0):
    """Sweep one sigma over sigma_grid; for each value run N_MC reps.
    Returns list of dicts with sigma_value, mse_mean, mse_std, dprime_mean, dprime_std.
    """
    results = []
    for sig_idx, sigma_val in enumerate(sigma_grid):
        sigmas = dict(fixed_sigmas)
        sigmas[sigma_name] = sigma_val
        mse_per_rep    = []
        dprime_per_rep = []

        for rep in trange(N_MC, desc=f"{sigma_name}={sigma_val:.4g}", leave=False):
            rep_mse = []
            rep_dp  = []
            for isi_val, exps in exps_by_isi.items():
                if not exps: continue
                hc_idx   = isi_to_hc_idx.get(isi_val)
                if hc_idx is None: continue
                human_dp = human_curve[hc_idx]
                run_out  = run_experiment_scores(
                    sigma0=sigmas["sigma0"], sigma1=sigmas["sigma1"], sigma2=sigmas["sigma2"],
                    t_step=t_step, rate=0, noise_mode=noise_mode,
                    metric=metric, X0=X, name_to_idx=name_to_idx,
                    experiment_list=exps, debug=False,
                    seed=base_seed + isi_val*1_000_000 + sig_idx*10_000 + rep,
                )
                hits = np.asarray(run_out["hits"]); fas = np.asarray(run_out["fas"])
                if len(hits)==0 or len(fas)==0: continue
                y = np.concatenate([np.ones(len(hits)), np.zeros(len(fas))])
                dp = auc_to_dprime(roc_auc_score(y, -np.concatenate([hits,fas])))
                rep_mse.append((dp - human_dp)**2)
                rep_dp.append(dp)
            if rep_mse:
                mse_per_rep.append(np.mean(rep_mse))
                dprime_per_rep.append(np.mean(rep_dp))

        results.append({
            "sigma_value":  sigma_val,
            "mse_mean":     np.mean(mse_per_rep)    if mse_per_rep    else np.nan,
            "mse_std":      np.std(mse_per_rep)     if mse_per_rep    else np.nan,
            "dprime_mean":  np.mean(dprime_per_rep) if dprime_per_rep else np.nan,
            "dprime_std":   np.std(dprime_per_rep)  if dprime_per_rep else np.nan,
        })
    return results


## Section A: Per-ISI Stability

Each ISI tested independently. Vary n_experiments in [20, 40, 80, 160].

In [ ]:
EXP_COUNTS = [20, 40, 80]
TEST_ISIS  = [1, 2, 4]
per_isi_results = {}   # isi -> {n_exp -> list_of_result_dicts}

for isi_val in TEST_ISIS:
    hc_idx = isi_to_hc_idx[isi_val]
    print(f'\n=== ISI={isi_val}  human d\' = {human_curve[hc_idx]:.4f} ===')
    n_exp_results = {}

    for n_exp in EXP_COUNTS:
        exps = make_toy_experiment_list(
            stimulus_pool, isi=isi_val, n_experiments=n_exp,
            k_stimuli=K_STIM, seed=isi_val*1000+n_exp)
        print(f'  n_exp={n_exp}: {len(exps)} exps, '
              f'avg len {np.mean([len(e) for e in exps]):.1f}')

        res = run_sigma_sweep(
            sigma_name='sigma1', sigma_grid=sigma1_grid,
            fixed_sigmas={'sigma0': sigma0_fixed, 'sigma1': 0, 'sigma2': sigma2_fixed},
            exps_by_isi={isi_val: exps},
            isi_to_hc_idx=isi_to_hc_idx, human_curve=human_curve,
            N_MC=N_MC, t_step=t_step, noise_mode=noise_mode,
            metric=metric, X=X, name_to_idx=name_to_idx,
            base_seed=isi_val*100_000_000 + n_exp*1_000_000,
        )
        n_exp_results[n_exp] = res

    per_isi_results[isi_val] = n_exp_results

print('Done.')



=== ISI=1  human d' = 2.9840 ===
  n_exp=20: 20 exps, avg len 8.0


sigma1=0.04793:   0%|          | 0/32 [00:00<?, ?it/s]

sigma1=0.08364:   0%|          | 0/32 [00:00<?, ?it/s]

sigma1=0.1459:   0%|          | 0/32 [00:00<?, ?it/s]

sigma1=0.2547:   0%|          | 0/32 [00:00<?, ?it/s]

...

sigma1=0.2547:   0%|          | 0/32 [00:00<?, ?it/s]

sigma1=0.4444:   0%|          | 0/32 [00:00<?, ?it/s]

sigma1=0.7755:   0%|          | 0/32 [00:00<?, ?it/s]

sigma1=1.353:   0%|          | 0/32 [00:00<?, ?it/s]

sigma1=2.361:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, isi_val in zip(axes, TEST_ISIS):
    hc_idx = isi_to_hc_idx[isi_val]
    for n_exp, res in per_isi_results[isi_val].items():
        df = pd.DataFrame(res)
        ax.plot(df.sigma_value, df.mse_mean, 'o-', label=f'N={n_exp}')
        ax.fill_between(df.sigma_value, df.mse_mean-df.mse_std,
                        df.mse_mean+df.mse_std, alpha=0.2)
    ax.set_xscale('log')
    ax.set_xlabel(r'$\sigma_1$')
    ax.set_ylabel('MSE')
    ax.set_title(f'ISI={isi_val}  (human d\u2032={human_curve[hc_idx]:.2f})')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.suptitle(f'Sigma1 per-ISI stability  ({encoder_type}-{layer}-{time_avg_tag})', y=1.03)
plt.tight_layout(); plt.show()

# d' version
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, isi_val in zip(axes, TEST_ISIS):
    hc_idx = isi_to_hc_idx[isi_val]
    for n_exp, res in per_isi_results[isi_val].items():
        df = pd.DataFrame(res)
        ax.errorbar(df.sigma_value, df.dprime_mean, yerr=df.dprime_std,
                    fmt='o-', capsize=3, label=f'N={n_exp}')
    ax.axhline(human_curve[hc_idx], color='k', ls='--', label='Human', alpha=0.6)
    ax.set_xscale('log'); ax.set_xlabel(r'$\sigma_1$'); ax.set_ylabel("d'")
    ax.set_title(f'ISI={isi_val}'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Section B: Combined ISI Stability

All ISIs [1, 2, 4] together — matching Stage B of `three_stage_fit`.

In [28]:
COMBINED_ISIS      = [1, 2, 4]
COMBINED_EXP_COUNTS = [10, 20, 40, 80]
combined_results = {}

for n_exp in COMBINED_EXP_COUNTS:
    print(f'\n--- n_exp/ISI={n_exp} ---')
    exps_by_isi = make_multi_isi_toy_experiments(
        stimulus_pool, isi_values=COMBINED_ISIS,
        n_experiments_per_isi=n_exp, k_stimuli=K_STIM, seed=42+n_exp)
    for iv, el in exps_by_isi.items():
        print(f'  ISI {iv}: {len(el)} exps, avg len {np.mean([len(e) for e in el]):.1f}')

    res = run_sigma_sweep(
        sigma_name='sigma1', sigma_grid=sigma1_grid,
        fixed_sigmas={'sigma0': sigma0_fixed, 'sigma1': 0, 'sigma2': sigma2_fixed},
        exps_by_isi=exps_by_isi,
        isi_to_hc_idx=isi_to_hc_idx, human_curve=human_curve,
        N_MC=N_MC, t_step=t_step, noise_mode=noise_mode,
        metric=metric, X=X, name_to_idx=name_to_idx,
        base_seed=500_000_000 + n_exp*1_000_000,
    )
    combined_results[n_exp] = res
print('Done.')



--- n_exp/ISI=10 ---
  ISI 1: 10 exps, avg len 8.0
  ISI 2: 10 exps, avg len 6.0
  ISI 4: 10 exps, avg len 10.0


sigma1=0.004793:   0%|          | 0/32 [00:00<?, ?it/s]

sigma1=0.01286:   0%|          | 0/32 [00:00<?, ?it/s]

sigma1=0.0345:   0%|          | 0/32 [00:00<?, ?it/s]

sigma1=0.09254:   0%|          | 0/32 [00:00<?, ?it/s]

...

sigma1=4.793:   0%|          | 0/32 [00:00<?, ?it/s]


--- n_exp/ISI=80 ---
  ISI 1: 80 exps, avg len 8.0
  ISI 2: 80 exps, avg len 6.0
  ISI 4: 80 exps, avg len 10.0


sigma1=0.004793:   0%|          | 0/32 [00:00<?, ?it/s]

sigma1=0.01286:   0%|          | 0/32 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(8, 5))
for n_exp, res in combined_results.items():
    df = pd.DataFrame(res)
    plt.plot(df.sigma_value, df.mse_mean, 'o-', label=f'N exp/ISI={n_exp}')
    plt.fill_between(df.sigma_value, df.mse_mean-df.mse_std,
                     df.mse_mean+df.mse_std, alpha=0.2)
plt.xscale('log'); plt.xlabel(r'$\sigma_1$'); plt.ylabel('Mean MSE [ISIs 1,2,4]')
plt.title(f'Sigma1 combined-ISI stability  N_MC={N_MC}, K={K_STIM}')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Section C: Which ISI Subsets Are Most Informative?

Fixed n_exp=40, compare all subsets of [1,2,4].

In [ ]:
ISI_SUBSETS = [[1],[2],[4],[1,2],[2,4],[1,4],[1,2,4]]
N_EXP_FIXED = 80
subset_results = {}

for subset in ISI_SUBSETS:
    print(f'--- subset={subset} ---')
    exps_by_isi = make_multi_isi_toy_experiments(
        stimulus_pool, isi_values=subset,
        n_experiments_per_isi=N_EXP_FIXED, k_stimuli=K_STIM,
        seed=sum(subset)*100+7)
    res = run_sigma_sweep(
        sigma_name='sigma1', sigma_grid=sigma1_grid,
        fixed_sigmas={'sigma0': sigma0_fixed, 'sigma1': 0, 'sigma2': sigma2_fixed},
        exps_by_isi=exps_by_isi,
        isi_to_hc_idx=isi_to_hc_idx, human_curve=human_curve,
        N_MC=N_MC, t_step=t_step, noise_mode=noise_mode,
        metric=metric, X=X, name_to_idx=name_to_idx,
        base_seed=sum(subset)*100_000_000,
    )
    subset_results[tuple(subset)] = res
print('Done.')


In [ ]:
plt.figure(figsize=(9, 6))
for subset_key, res in subset_results.items():
    df = pd.DataFrame(res)
    plt.plot(df.sigma_value, df.mse_mean, 'o-', label=f'ISIs {list(subset_key)}')
    plt.fill_between(df.sigma_value, df.mse_mean-df.mse_std,
                     df.mse_mean+df.mse_std, alpha=0.15)
plt.xscale('log'); plt.xlabel(r'$\sigma_1$'); plt.ylabel('Mean MSE')
plt.title(f'Which ISI subsets are most informative?  (n={N_EXP_FIXED}/ISI, N_MC={N_MC})')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Summary

*(Fill in after running)*

**Recommended Stage B settings:**

In [ ]:
print('Cost estimates (run_experiment_scores calls per fit):')
print(f'  Current: n_grid=15, n_mc=32, n_refine=2, 3 ISIs = {15*32*3*2}')
for n_isi in [1,2,3]:
    for n_mc in [16,32]:
        for n_grid in [10,15]:
            print(f'  {n_isi} ISIs n_grid={n_grid} n_mc={n_mc}: {n_grid*n_mc*n_isi*2}')
